# AI Programming — Lecture 8
## Training Neural Networks

이 노트북에서는 Lecture 8의 세 가지 핵심 주제를 직접 확인합니다.

1. **Gradient-based Optimizers**
2. **Learning Rate Control**
3. **Parameter Initialization**

복잡한 모델의 성능 경쟁보다는, 같은 문제에서 설정을 바꾸었을 때
**학습 과정이 어떻게 달라지는지 관찰하는 것**이 목표입니다.

### 학습 목표

실습을 마치면 다음 내용을 설명할 수 있어야 합니다.

- Full-batch GD와 mini-batch SGD의 차이를 설명할 수 있습니다.
- Momentum이 SGD의 진동을 완화하는 이유를 이해합니다.
- Adam이 momentum과 adaptive step size를 결합한다는 점을 이해합니다.
- Learning rate가 너무 작거나 너무 클 때 나타나는 현상을 관찰할 수 있습니다.
- Constant LR, cosine decay, warm-up + cosine decay의 차이를 설명할 수 있습니다.
- Weight initialization이 activation scale에 영향을 준다는 점을 확인할 수 있습니다.
- Xavier initialization이 `tanh`에, He initialization이 `ReLU`에 적합한 이유를 직관적으로 이해합니다.

### 실습 방법

1. 셀을 위에서부터 순서대로 실행하세요.
2. 각 절의 **확인할 내용**을 읽고 그래프를 해석하세요.
3. `TODO`가 표시된 값은 직접 바꾸어 다시 실행하세요.
4. 이번 실습에서는 최종 loss 하나보다 **loss curve와 parameter update의 형태**를 관찰하는 것이 중요합니다.

## 0. 라이브러리 불러오기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)

# Part I. Gradient-Based Optimizers

## 1. Lecture 8의 작은 Regression Dataset

슬라이드에서 사용한 다음 데이터를 그대로 사용합니다.

| Study Time ($x$) | 1 | 3 | 5 | 7 | 9 | 11 |
|---|---:|---:|---:|---:|---:|---:|
| Exam Score ($y$) | 75 | 77 | 85 | 83 | 90 | 91 |

모델은 가장 단순한 선형 모델을 사용합니다.

$$
\hat{y} = wx+b
$$

Optimizer의 차이에만 집중하기 위해 입력과 출력은 평균 0, 표준편차 1이 되도록
표준화한 뒤 학습합니다.

In [ ]:
x_raw = np.array([1, 3, 5, 7, 9, 11], dtype=float)
y_raw = np.array([75, 77, 85, 83, 90, 91], dtype=float)

x_mean, x_std = x_raw.mean(), x_raw.std()
y_mean, y_std = y_raw.mean(), y_raw.std()

x = (x_raw - x_mean) / x_std
y = (y_raw - y_mean) / y_std

print("x:", x)
print("y:", y)

In [ ]:
plt.scatter(x_raw, y_raw)
plt.xlabel("Study Time")
plt.ylabel("Exam Score")
plt.title("Lecture 8 Regression Dataset")
plt.grid(alpha=0.3)
plt.show()

## 2. Loss와 Gradient

MSE를 사용합니다.

$$
\mathcal{L}
=
\frac{1}{N}
\sum_i
(\hat{y}_i-y_i)^2
$$

선형 모델 $\hat{y}=wx+b$의 gradient는 다음과 같이 계산할 수 있습니다.

$$
\frac{\partial \mathcal{L}}{\partial w}
=
\frac{2}{N}
\sum_i
(\hat{y}_i-y_i)x_i
$$

$$
\frac{\partial \mathcal{L}}{\partial b}
=
\frac{2}{N}
\sum_i
(\hat{y}_i-y_i)
$$

In [ ]:
def mse_and_grad(w, b, x_batch, y_batch):
    y_hat = w * x_batch + b
    error = y_hat - y_batch

    loss = np.mean(error ** 2)

    grad_w = 2.0 * np.mean(error * x_batch)
    grad_b = 2.0 * np.mean(error)

    return loss, grad_w, grad_b

## 3. Full-Batch Gradient Descent

Full-batch GD는 **전체 training data**를 사용해 한 번 gradient를 계산합니다.

따라서 한 epoch에 parameter update가 한 번 발생합니다.

In [ ]:
def train_full_batch_gd(
    x,
    y,
    lr=0.05,
    epochs=100,
):
    w = 0.0
    b = 0.0

    losses = []
    update_steps = []

    for epoch in range(epochs):
        loss, grad_w, grad_b = mse_and_grad(
            w, b, x, y
        )

        w -= lr * grad_w
        b -= lr * grad_b

        losses.append(loss)
        update_steps.append((w, b))

    return w, b, np.array(losses), np.array(update_steps)


w_gd, b_gd, loss_gd, path_gd = train_full_batch_gd(
    x,
    y,
    lr=0.05,
    epochs=100,
)

print("Final w:", w_gd)
print("Final b:", b_gd)
print("Number of updates:", len(path_gd))

## 4. Mini-Batch SGD

Mini-batch SGD는 전체 데이터가 아니라 **일부 sample**로 gradient를 계산합니다.

이번에는 슬라이드와 같이 `batch_size=2`를 사용합니다.

6개의 sample을 2개씩 사용하므로 한 epoch에 3번 parameter가 업데이트됩니다.

In [ ]:
def train_minibatch_sgd(
    x,
    y,
    lr=0.05,
    epochs=100,
    batch_size=2,
    seed=0,
):
    rng = np.random.default_rng(seed)

    w = 0.0
    b = 0.0

    epoch_losses = []
    update_steps = []

    n = len(x)

    for epoch in range(epochs):
        indices = rng.permutation(n)

        for start in range(0, n, batch_size):
            batch_idx = indices[
                start:start + batch_size
            ]

            xb = x[batch_idx]
            yb = y[batch_idx]

            _, grad_w, grad_b = mse_and_grad(
                w, b, xb, yb
            )

            w -= lr * grad_w
            b -= lr * grad_b

            update_steps.append((w, b))

        full_loss, _, _ = mse_and_grad(
            w, b, x, y
        )
        epoch_losses.append(full_loss)

    return (
        w,
        b,
        np.array(epoch_losses),
        np.array(update_steps),
    )


w_sgd, b_sgd, loss_sgd, path_sgd = train_minibatch_sgd(
    x,
    y,
    lr=0.05,
    epochs=100,
    batch_size=2,
    seed=0,
)

print("Final w:", w_sgd)
print("Final b:", b_sgd)
print("Number of updates:", len(path_sgd))

In [ ]:
plt.plot(loss_gd, label="Full-batch GD")
plt.plot(loss_sgd, label="Mini-batch SGD")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("GD vs. Mini-Batch SGD")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 확인할 내용

- Full-batch GD: 한 epoch에 1번 update
- Mini-batch SGD: 한 epoch에 여러 번 update
- SGD는 batch마다 gradient가 달라지므로 loss curve가 조금 더 noisy할 수 있습니다.

## 5. Momentum

Momentum은 과거 gradient 정보를 누적하여 update 방향을 부드럽게 만듭니다.

개념적으로는

$$
v_t
=
\beta v_{t-1}
+
g_t
$$

$$
w_t
=
w_{t-1}
-
\eta v_t
$$

처럼 생각할 수 있습니다.

Lecture 8의 핵심은 **진동을 줄이고 같은 방향의 움직임을 가속**한다는 점입니다.

In [ ]:
def train_momentum(
    x,
    y,
    lr=0.05,
    momentum=0.9,
    epochs=100,
    batch_size=2,
    seed=0,
):
    rng = np.random.default_rng(seed)

    w = 0.0
    b = 0.0

    v_w = 0.0
    v_b = 0.0

    epoch_losses = []

    n = len(x)

    for epoch in range(epochs):
        indices = rng.permutation(n)

        for start in range(0, n, batch_size):
            batch_idx = indices[
                start:start + batch_size
            ]

            xb = x[batch_idx]
            yb = y[batch_idx]

            _, grad_w, grad_b = mse_and_grad(
                w, b, xb, yb
            )

            v_w = momentum * v_w + grad_w
            v_b = momentum * v_b + grad_b

            w -= lr * v_w
            b -= lr * v_b

        full_loss, _, _ = mse_and_grad(
            w, b, x, y
        )
        epoch_losses.append(full_loss)

    return w, b, np.array(epoch_losses)


w_mom, b_mom, loss_mom = train_momentum(
    x,
    y,
    lr=0.02,
    momentum=0.9,
    epochs=100,
    batch_size=2,
)

print("Final w:", w_mom)
print("Final b:", b_mom)

## 6. Adam

Adam은 Lecture 8에서 설명한 것처럼

- **First moment**: gradient의 이동 평균
- **Second moment**: squared gradient의 이동 평균
- **Bias correction**
- **Parameter-wise adaptive step size**

를 결합합니다.

이번에는 가장 단순한 형태로 직접 구현하여 흐름만 확인합니다.

In [ ]:
def train_adam(
    x,
    y,
    lr=0.05,
    beta1=0.9,
    beta2=0.999,
    eps=1e-8,
    epochs=100,
    batch_size=2,
    seed=0,
):
    rng = np.random.default_rng(seed)

    w = 0.0
    b = 0.0

    m_w = 0.0
    m_b = 0.0

    v_w = 0.0
    v_b = 0.0

    t = 0
    epoch_losses = []

    n = len(x)

    for epoch in range(epochs):
        indices = rng.permutation(n)

        for start in range(0, n, batch_size):
            batch_idx = indices[
                start:start + batch_size
            ]

            xb = x[batch_idx]
            yb = y[batch_idx]

            _, grad_w, grad_b = mse_and_grad(
                w, b, xb, yb
            )

            t += 1

            # First moment
            m_w = beta1 * m_w + (1 - beta1) * grad_w
            m_b = beta1 * m_b + (1 - beta1) * grad_b

            # Second moment
            v_w = beta2 * v_w + (1 - beta2) * (grad_w ** 2)
            v_b = beta2 * v_b + (1 - beta2) * (grad_b ** 2)

            # Bias correction
            m_w_hat = m_w / (1 - beta1 ** t)
            m_b_hat = m_b / (1 - beta1 ** t)

            v_w_hat = v_w / (1 - beta2 ** t)
            v_b_hat = v_b / (1 - beta2 ** t)

            # Parameter update
            w -= lr * m_w_hat / (np.sqrt(v_w_hat) + eps)
            b -= lr * m_b_hat / (np.sqrt(v_b_hat) + eps)

        full_loss, _, _ = mse_and_grad(
            w, b, x, y
        )
        epoch_losses.append(full_loss)

    return w, b, np.array(epoch_losses)


w_adam, b_adam, loss_adam = train_adam(
    x,
    y,
    lr=0.05,
    epochs=100,
    batch_size=2,
)

print("Final w:", w_adam)
print("Final b:", b_adam)

In [ ]:
plt.plot(loss_gd, label="Full-batch GD")
plt.plot(loss_sgd, label="Mini-batch SGD")
plt.plot(loss_mom, label="Momentum")
plt.plot(loss_adam, label="Adam")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Optimizer Comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 직접 해보기 1 — Optimizer

다음 값을 바꾸어 보세요.

- SGD `batch_size = 1, 2, 6`
- Momentum `momentum = 0.0, 0.5, 0.9`
- Adam `lr = 0.005, 0.05, 0.2`

Loss curve의 속도와 안정성이 어떻게 달라지는지 비교하세요.

### 참고: Keras에서 Optimizer 지정하기

Lecture 8의 optimizer들은 실제 Keras에서는 다음처럼 사용할 수 있습니다.

In [ ]:
from tensorflow.keras.optimizers import (
    SGD,
    Adagrad,
    RMSprop,
    Adam,
    AdamW,
)

optimizer_gd = SGD(learning_rate=0.01)
optimizer_momentum = SGD(
    learning_rate=0.01,
    momentum=0.9,
    nesterov=True,
)
optimizer_adagrad = Adagrad(learning_rate=0.01)
optimizer_rmsprop = RMSprop(learning_rate=0.001)
optimizer_adam = Adam(learning_rate=0.001)
optimizer_adamw = AdamW(
    learning_rate=0.001,
    weight_decay=1e-4,
)

print("Optimizers created.")

> Lecture 8에서 다룬 Newton-type second-order method는 Hessian 계산 비용이 매우 크므로,
> 이 노트북에서는 실제 구현하지 않고 개념 수준으로만 남겨 둡니다.

# Part II. Learning Rate Control

## 7. Learning Rate가 너무 작거나 크면?

Gradient descent update는

$$
w
\leftarrow
w
-
\eta
\frac{\partial \mathcal{L}}
{\partial w}
$$

입니다.

$\eta$가 learning rate입니다.

Lecture 8의 핵심:

```text
Too small → 느림
Good      → 안정적으로 감소
Too large → 불안정하거나 발산
```

In [ ]:
def train_with_lr(
    x,
    y,
    lr,
    epochs=40,
):
    w = 0.0
    b = 0.0
    losses = []

    for _ in range(epochs):
        loss, grad_w, grad_b = mse_and_grad(
            w, b, x, y
        )

        losses.append(loss)

        w -= lr * grad_w
        b -= lr * grad_b

        if (
            not np.isfinite(loss)
            or loss > 1e6
        ):
            break

    return np.array(losses)


loss_lr_small = train_with_lr(
    x, y, lr=0.001
)

loss_lr_good = train_with_lr(
    x, y, lr=0.05
)

loss_lr_large = train_with_lr(
    x, y, lr=1.2
)

In [ ]:
plt.plot(loss_lr_small, label="Small LR = 0.001")
plt.plot(loss_lr_good, label="Good LR = 0.05")
plt.plot(loss_lr_large, label="Very Large LR = 1.2")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Effect of Learning Rate")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 확인할 내용

- 작은 LR은 loss를 줄이지만 속도가 느립니다.
- 적절한 LR은 빠르고 안정적으로 감소합니다.
- 너무 큰 LR은 optimum을 지나치며 불안정하거나 발산할 수 있습니다.

Learning rate의 적절한 범위는 문제와 batch size에 따라 달라집니다.

## 8. Learning Rate Scheduling

Lecture 8에서는 다음 흐름을 소개합니다.

```text
Warm-up → (Constant) → Decay
```

특히 널리 사용되는 형태는

```text
Warm-up + Cosine Decay
```

입니다.

- **Warm-up**: 학습 초기에 LR을 천천히 증가
- **Decay**: 수렴에 가까워질수록 update를 작게 만듦

In [ ]:
total_steps = 100
warmup_steps = 10
base_lr = 1e-3

steps = np.arange(total_steps)

constant_lr = np.full(
    total_steps,
    base_lr,
)

cosine_lr = (
    base_lr
    * 0.5
    * (
        1
        + np.cos(
            np.pi * steps / (total_steps - 1)
        )
    )
)

warmup_cosine_lr = np.zeros(total_steps)

for step in steps:
    if step < warmup_steps:
        warmup_cosine_lr[step] = (
            base_lr
            * step
            / warmup_steps
        )
    else:
        progress = (
            step - warmup_steps
        ) / (
            total_steps - warmup_steps - 1
        )

        warmup_cosine_lr[step] = (
            base_lr
            * 0.5
            * (
                1
                + np.cos(np.pi * progress)
            )
        )

In [ ]:
plt.plot(
    steps,
    constant_lr,
    label="Constant"
)
plt.plot(
    steps,
    cosine_lr,
    label="Cosine Decay"
)
plt.plot(
    steps,
    warmup_cosine_lr,
    label="Warm-up + Cosine"
)
plt.xlabel("Training Step")
plt.ylabel("Learning Rate")
plt.title("Learning Rate Schedules")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 직접 해보기 2 — Learning Rate Schedule

다음을 바꾸어 보세요.

```python
warmup_steps = 5
warmup_steps = 20
base_lr = 1e-4
base_lr = 1e-2
```

Warm-up 구간과 decay curve가 어떻게 달라지는지 확인하세요.

### Keras의 Cosine Decay 예제

슬라이드에서 소개한 `CosineDecay`를 그대로 사용할 수 있습니다.

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay

initial_learning_rate = 0.001

lr_schedule = CosineDecay(
    initial_learning_rate,
    decay_steps=100000,
    alpha=0.0,
)

optimizer = Adam(
    learning_rate=lr_schedule
)

print("Adam + CosineDecay created.")

# Part III. Parameter Initialization

## 9. 왜 Initialization이 중요한가?

신경망의 각 layer는 반복적으로

$$
\mathbf{z}
=
\mathbf{W}\mathbf{h}
+
\mathbf{b}
$$

를 계산합니다.

Weight scale이 잘못되면 layer가 깊어질수록 activation이

- 너무 작아지거나
- 너무 커지거나
- `tanh`에서 saturation되거나
- ReLU에서 scale이 빠르게 줄어들 수 있습니다.

Lecture 8의 핵심은 **activation과 gradient의 scale을 layer를 지나면서 가능한 안정적으로 유지하는 것**입니다.

## 10. Zero Initialization

Hidden layer의 모든 neuron을 똑같이 0으로 초기화하면
같은 입력에 대해 같은 출력을 만들게 됩니다.

따라서 neuron들이 서로 다른 feature를 학습하기 어렵습니다.

아래에서는 두 hidden unit가 완전히 같은 값을 만드는 것을 확인합니다.

In [ ]:
x_demo = np.array([1.0, 2.0, 3.0])

W_zero = np.zeros((3, 2))
b_zero = np.zeros(2)

z_zero = x_demo @ W_zero + b_zero

print("Hidden unit outputs:", z_zero)
print(
    "Two units are identical:",
    z_zero[0] == z_zero[1]
)

## 11. Layer별 Activation Scale 측정

6-layer network를 만들고 각 layer의 activation 표준편차를 기록합니다.

실습에서는 차이가 잘 보이도록 작은/큰 random scale을 선택합니다.

Xavier와 He는 fan-in을 고려하여 weight scale을 정합니다.

### Xavier

대표적으로

$$
\mathrm{std}
\approx
\sqrt{\frac{1}{fan_{in}}}
$$

형태로 생각할 수 있습니다.

### He

ReLU에서는 절반 정도의 activation이 0이 되기 때문에 더 큰 variance를 사용합니다.

$$
\mathrm{std}
\approx
\sqrt{\frac{2}{fan_{in}}}
$$

In [ ]:
def relu(x):
    return np.maximum(0.0, x)

def initialize_weight(
    fan_in,
    fan_out,
    method,
    rng,
):
    if method == "small":
        std = 0.01

    elif method == "large":
        # 차이를 쉽게 관찰하기 위한 실습용 값
        std = 0.10

    elif method == "xavier":
        std = np.sqrt(1.0 / fan_in)

    elif method == "he":
        std = np.sqrt(2.0 / fan_in)

    else:
        raise ValueError(
            "Unknown initialization method"
        )

    return rng.normal(
        0.0,
        std,
        size=(fan_in, fan_out),
    )


def forward_activation_stats(
    method,
    activation,
    depth=6,
    width=128,
    n_samples=1000,
    seed=0,
):
    rng = np.random.default_rng(seed)

    h = rng.normal(
        0.0,
        1.0,
        size=(n_samples, width),
    )

    activations = []

    for _ in range(depth):
        W = initialize_weight(
            fan_in=width,
            fan_out=width,
            method=method,
            rng=rng,
        )

        z = h @ W

        if activation == "tanh":
            h = np.tanh(z)

        elif activation == "relu":
            h = relu(z)

        else:
            raise ValueError(
                "Unknown activation"
            )

        activations.append(h.copy())

    means = np.array([
        a.mean()
        for a in activations
    ])

    stds = np.array([
        a.std()
        for a in activations
    ])

    return activations, means, stds

## 12. Small Random Initialization + `tanh`

작은 weight를 사용하면 activation이 layer를 지나면서 0 근처로 수축할 수 있습니다.

In [ ]:
act_small, mean_small, std_small = (
    forward_activation_stats(
        method="small",
        activation="tanh",
    )
)

print("Layer means:", mean_small)
print("Layer stds :", std_small)

In [ ]:
layers = np.arange(1, 7)

plt.plot(
    layers,
    std_small,
    marker="o",
)
plt.xlabel("Layer")
plt.ylabel("Activation Std")
plt.title("Small Random Initialization + tanh")
plt.grid(alpha=0.3)
plt.show()

## 13. Large Random Initialization + `tanh`

큰 weight를 사용하면 `tanh` 입력의 절댓값이 커져
activation이 `-1` 또는 `1` 근처로 saturation될 수 있습니다.

> 슬라이드의 예제와 같은 현상을 더 쉽게 관찰하기 위해 이 실습에서는 `std=0.10`을 사용합니다.
> `0.05`로 바꾸어 비교해 보세요.

In [ ]:
act_large, mean_large, std_large = (
    forward_activation_stats(
        method="large",
        activation="tanh",
    )
)

print("Layer means:", mean_large)
print("Layer stds :", std_large)

last_layer = act_large[-1].ravel()

plt.hist(
    last_layer,
    bins=50,
)
plt.xlabel("Activation")
plt.ylabel("Count")
plt.title("Large Random + tanh: Layer 6")
plt.grid(alpha=0.3)
plt.show()

## 14. Xavier Initialization + `tanh`

Xavier initialization은 `sigmoid`나 `tanh`와 같이
양쪽으로 값을 전달하는 activation에서 forward activation scale을 비교적 안정적으로 유지하도록 설계되었습니다.

In [ ]:
act_xavier_tanh, mean_xavier_tanh, std_xavier_tanh = (
    forward_activation_stats(
        method="xavier",
        activation="tanh",
    )
)

print("Layer means:", mean_xavier_tanh)
print("Layer stds :", std_xavier_tanh)

In [ ]:
plt.plot(
    layers,
    std_xavier_tanh,
    marker="o",
)
plt.xlabel("Layer")
plt.ylabel("Activation Std")
plt.title("Xavier Initialization + tanh")
plt.grid(alpha=0.3)
plt.show()

## 15. Xavier Initialization + `ReLU`

Lecture 8에서는 Xavier가 `tanh`에서는 잘 동작하지만,
ReLU에서는 activation scale이 감소할 수 있음을 보여줍니다.

In [ ]:
act_xavier_relu, mean_xavier_relu, std_xavier_relu = (
    forward_activation_stats(
        method="xavier",
        activation="relu",
    )
)

print("Layer means:", mean_xavier_relu)
print("Layer stds :", std_xavier_relu)

In [ ]:
plt.plot(
    layers,
    std_xavier_relu,
    marker="o",
)
plt.xlabel("Layer")
plt.ylabel("Activation Std")
plt.title("Xavier Initialization + ReLU")
plt.grid(alpha=0.3)
plt.show()

## 16. He Initialization + `ReLU`

He initialization은 ReLU에서 절반 정도의 activation이 0이 되는 특성을 고려하여
Xavier보다 큰 초기 variance를 사용합니다.

Lecture 8의 핵심 비교:

```text
Xavier + tanh → relatively stable
Xavier + ReLU → activations may shrink
He + ReLU     → more stable activation scale
```

In [ ]:
act_he_relu, mean_he_relu, std_he_relu = (
    forward_activation_stats(
        method="he",
        activation="relu",
    )
)

print("Layer means:", mean_he_relu)
print("Layer stds :", std_he_relu)

In [ ]:
plt.plot(
    layers,
    std_he_relu,
    marker="o",
)
plt.xlabel("Layer")
plt.ylabel("Activation Std")
plt.title("He Initialization + ReLU")
plt.grid(alpha=0.3)
plt.show()

## 17. Initialization 비교

In [ ]:
plt.plot(
    layers,
    std_small,
    marker="o",
    label="Small + tanh",
)
plt.plot(
    layers,
    std_xavier_tanh,
    marker="o",
    label="Xavier + tanh",
)
plt.plot(
    layers,
    std_xavier_relu,
    marker="o",
    label="Xavier + ReLU",
)
plt.plot(
    layers,
    std_he_relu,
    marker="o",
    label="He + ReLU",
)
plt.xlabel("Layer")
plt.ylabel("Activation Std")
plt.title("Activation Scale Across Layers")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 확인할 내용

- 작은 random initialization에서는 activation scale이 빠르게 줄어드는지 확인하세요.
- Xavier + `tanh`가 작은 random보다 안정적인지 확인하세요.
- Xavier + ReLU에서는 layer가 깊어질수록 scale이 감소하는지 확인하세요.
- He + ReLU가 ReLU network에서 scale을 더 잘 유지하는지 확인하세요.

실제 값은 random seed, network width, input distribution에 따라 달라질 수 있습니다.
중요한 것은 **layer depth에 따른 상대적인 변화**입니다.

### 직접 해보기 3 — Initialization

다음을 바꾸어 보세요.

1. `depth=6 → 12`
2. `width=128 → 256`
3. large random initialization의 `std=0.10 → 0.05`
4. Xavier + ReLU와 He + ReLU 비교
5. `tanh` 대신 ReLU를 사용할 때 activation histogram이 어떻게 달라지는지 확인

# 18. 최종 실습

### Optimizer

1. Full-batch GD와 mini-batch SGD의 update 횟수를 비교하세요.
2. `batch_size=1, 2, 6`으로 SGD를 학습하고 loss curve를 비교하세요.
3. Momentum coefficient를 `0.0, 0.5, 0.9`로 변경해 보세요.
4. Adam의 learning rate를 바꾸어 수렴 속도를 비교하세요.

### Learning Rate

5. 너무 작은 LR, 적절한 LR, 너무 큰 LR을 각각 사용해 보세요.
6. Warm-up step 수를 바꾸어 schedule의 모양을 확인하세요.
7. Constant LR과 cosine decay를 비교하세요.

### Initialization

8. 작은 random initialization에서 activation std가 layer별로 어떻게 변하는지 확인하세요.
9. 큰 random initialization에서 `tanh` saturation을 histogram으로 확인하세요.
10. Xavier + tanh와 Xavier + ReLU를 비교하세요.
11. Xavier + ReLU와 He + ReLU를 비교하세요.

### 생각해 보기

12. Optimizer를 바꾸는 것과 learning rate를 바꾸는 것은 어떤 차이가 있습니까?
13. 좋은 optimizer를 사용하더라도 initialization이 나쁘면 학습이 어려울 수 있는 이유를 설명하세요.

# 19. 정리

Lecture 8의 핵심을 하나의 training pipeline으로 정리하면 다음과 같습니다.

```text
Parameter Initialization
        ↓
Forward Pass
        ↓
Loss
        ↓
Backpropagation
        ↓
Optimizer
        ↓
Learning Rate Schedule
        ↓
Parameter Update
        ↓
Repeat
```

### Optimizer

| Method | 핵심 아이디어 |
|---|---|
| GD | 전체 데이터로 정확한 gradient |
| SGD | mini-batch로 더 자주 update |
| Momentum | 과거 gradient 방향을 누적 |
| Adagrad | parameter별 adaptive LR |
| RMSProp | squared gradient의 moving average |
| Adam | Momentum + adaptive step size |
| AdamW | Adam + decoupled weight decay |

### Learning Rate

```text
Too small → slow
Good      → stable convergence
Too large → unstable / divergence
```

실전에서는 자주

```text
Warm-up → Cosine Decay
```

형태를 사용합니다.

### Initialization

```text
Too small       → activations shrink
Too large       → saturation
Xavier + tanh   → relatively stable
He + ReLU       → relatively stable
```

### 꼭 기억할 것

1. **Optimizer는 gradient를 이용해 parameter를 어떻게 움직일지를 결정합니다.**
2. **Learning rate는 한 번에 얼마나 크게 움직일지를 결정합니다.**
3. **Initialization은 training을 시작하는 activation과 gradient의 scale을 결정합니다.**